# **Semantic Web & Linked Open Data with Python**
## บทที่ 09 — Well-known Ontologies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch09/ch09_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   http://xmlns.com/foaf/spec/
*   https://www.w3.org/TR/skos-reference/
*   https://query.wikidata.org/
*   https://www.dbpedia.org/

📖 เอกสารประกอบบทนี้: [`09-well-known-ontologies/`](https://github.com/Rathachai/semantic-web/tree/main/09-well-known-ontologies)

---

### Setup

In [1]:
!pip install -q rdflib SPARQLWrapper

In [2]:
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import FOAF, RDF, RDFS, SKOS
from SPARQLWrapper import SPARQLWrapper, JSON

## FOAF — Friend of a Friend

vocabulary มาตรฐานสำหรับอธิบายบุคคลและความสัมพันธ์ทางสังคม

In [3]:
foaf_ttl = """
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex:   <http://example.org/people/> .

ex:alice a foaf:Person ;
    foaf:name "Alice" ;
    foaf:mbox <mailto:alice@example.org> ;
    foaf:knows ex:bob .
ex:bob a foaf:Person ;
    foaf:name "Bob" .
"""
g = Graph().parse(data=foaf_ttl, format="turtle")
print(g.serialize(format="turtle"))

@prefix ex: <http://example.org/people/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:alice a foaf:Person ;
    foaf:knows ex:bob ;
    foaf:mbox <mailto:alice@example.org> ;
    foaf:name "Alice" .

ex:bob a foaf:Person ;
    foaf:name "Bob" .




## SKOS — Simple Knowledge Organization System

ใช้จัดหมวดหมู่แบบ thesaurus ด้วย `broader`/`narrower`/`related`

In [4]:
skos_ttl = """
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex:   <http://example.org/topic/> .

ex:ComputerScience a skos:Concept ; skos:prefLabel "Computer Science"@en .
ex:MachineLearning a skos:Concept ;
    skos:prefLabel "Machine Learning"@en ;
    skos:broader ex:ComputerScience .
ex:DeepLearning a skos:Concept ;
    skos:prefLabel "Deep Learning"@en ;
    skos:broader ex:MachineLearning .
"""
g2 = Graph().parse(data=skos_ttl, format="turtle")

res = g2.query("""
    SELECT ?label WHERE {
      ex:DeepLearning skos:broader+ ?c .
      ?c skos:prefLabel ?label .
    }
""", initNs={"skos": SKOS, "ex": Namespace("http://example.org/topic/")})
for row in res:
    print(row.label)

Machine Learning
Computer Science


## Query ข้อมูลจาก Wikidata จริง

⚠️ Wikidata บล็อก request ที่ไม่ระบุ `User-Agent` — ต้องใส่ชื่อโครงการและช่องทางติดต่อเสมอ

In [5]:
UA = "SemanticWebCourse/1.0 (https://github.com/Rathachai/semantic-web)"

sparql = SPARQLWrapper("https://query.wikidata.org/sparql", agent=UA)
sparql.setQuery("""
    SELECT ?label ?population WHERE {
      wd:Q1861 rdfs:label ?label ; wdt:P1082 ?population .
      FILTER(LANG(?label) = "th")
    }
""")
sparql.setReturnFormat(JSON)
try:
    results = sparql.query().convert()
    for row in results["results"]["bindings"]:
        print(row["label"]["value"], "- population:", row["population"]["value"])
except Exception as exc:
    print(f"⚠️ เรียก Wikidata ไม่สำเร็จตอนนี้: {type(exc).__name__}")

กรุงเทพมหานคร - population: 5676648


## แปลงผลจาก Wikidata เป็น schema.org RDF ของเราเอง

In [6]:
SCHEMA = Namespace("https://schema.org/")
WD = Namespace("http://www.wikidata.org/entity/")

g3 = Graph()
g3.bind("schema", SCHEMA)
bangkok = URIRef("http://example.org/place/bangkok")
g3.add((bangkok, RDF.type, SCHEMA.City))
g3.add((bangkok, SCHEMA.name, Literal("Bangkok", lang="en")))
from rdflib.namespace import OWL
g3.add((bangkok, OWL.sameAs, WD.Q1861))
print(g3.serialize(format="turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix schema: <https://schema.org/> .

<http://example.org/place/bangkok> a schema:City ;
    owl:sameAs <http://www.wikidata.org/entity/Q1861> ;
    schema:name "Bangkok"@en .




## 🎯 สรุป

- **FOAF** อธิบายบุคคลและความสัมพันธ์ทางสังคม, **SKOS** จัดหมวดหมู่แบบ thesaurus
- `skos:broader+` (property path) ไล่ลำดับชั้นได้หลายขั้นในคำสั่งเดียว
- ⚠️ **ต้องระบุ `User-Agent` เสมอ** เมื่อเรียก Wikidata Query Service มิฉะนั้นถูกบล็อก
- ผูก vocabulary ของเราเข้ากับมาตรฐาน (`owl:sameAs` ไปยัง `wd:`) ทำให้คนอื่นเข้าใจโดยไม่ต้องอ่านเอกสาร

ลองเปลี่ยน `wd:Q1861` เป็น QID ของเมืองอื่น แล้วดูว่า query คืนอะไร!

---

## จบบทที่ 09 🎉

ไปต่อบทถัดไป: **บทที่ 10 — Ontology Design**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*